## model

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from typing import Annotated
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict
from langgraph.graph import MessagesState
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

#llm
##################

llm = init_chat_model("ollama:nemotron-3-super:cloud",base_url="https://ollama.com")
llmg = init_chat_model("google_genai:gemini-2.5-flash")

response = llmg.invoke("What is the color of the sky answer in one word?")
print(response.content)

Blue


## RAG

### html page extraction

In [ ]:
import bs4
import requests
from langchain_core.documents import Document

# Below is a minimal helper for demonstration purposes.
def load_web_page(url: str, bs_kwargs: dict | None = None) -> list[Document]:
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    soup = bs4.BeautifulSoup(response.text, "html.parser", **(bs_kwargs or {}))
    return [Document(page_content=soup.get_text(), metadata={"source": url})]


urls = [
    "https://lifeinsurance.adityabirlacapital.com/term-insurance/"
]

docs = [load_web_page(url) for url in urls]

In [ ]:
docs

[[Document(metadata={'source': 'https://lifeinsurance.adityabirlacapital.com/term-insurance/'}, page_content='Term Insurance | Term Insurance Plan | ABSLIABC HomeABC Solutions ConnectAll InsuranceEndowment PlansNewABSLI Anmol AkshayaNewHer Assurance - Endowment Plan for WomenNewABSLI Akshaya PlanABSLI Vision Endowment Plus PlanABSLI Vision LifeIncome PlanABSLI Vision LifeIncome Plus PlanSavings PlansNewHer Nishchit - Savings Plan for WomenNewABSLI Nishchit Laabh PlanABSLI Nishchit Aayush PlanABSLI Assured Income PlusABSLI SecurePlus PlanABSLI Assured Savings PlanABSLI Guaranteed Milestone PlanTerm InsuranceNewHer Care - Term Plan for WomenNewABSLI Super Term PlanNewABSLI Income Suraksha PlanABSLI Salaried Term PlanABSLI DigiShield PlanABSLI Poorna Suraksha KawachABSLI Insta Digi Term PlanABSLI Saral Jeevan BimaABSLI Super Term Plan PlusPension PlansNewABSLI Guaranteed Annuity PlusNewHer Retirement - Pension Plan for WomenNewABSLI Index Guaranteed Annuity PlusNewABSLI Nishchit Pension P

In [ ]:
print(docs[0][0].page_content)

### chunking

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_chroma import Chroma

# --------------------------------------------------
# Load PDF
# --------------------------------------------------

loader = PyPDFLoader("resources/Term Insurance _clean.pdf")
documents = loader.load()

# --------------------------------------------------
# Chunk
# --------------------------------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))

Chunks: 150


In [6]:
documents[:3]

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-06-28T11:56:25+05:30', 'title': 'Term Insurance | Term Insurance Plan | ABSLI', 'author': 'smrit', 'moddate': '2026-06-28T11:56:25+05:30', 'source': 'resources/Term Insurance _clean.pdf', 'total_pages': 27, 'page': 0, 'page_label': '1'}, page_content='28/06/2026, 11:15 Term Insurance | Term Insurance Plan | ABSLI \nhttps://lifeinsurance.adityabirlacapital.com/term-insurance/ 1/30 \n \n \n \n \n \n \nBUY  ONLINE  \n \n \nTerm Insurance \n \nWhat Is Term Insurance? \nTerm Insurance is a type of life insurance policy that provides ﬁnancial protection to your loved \nones in case term policy your untimely demise during the policy term. It is a pure protection plan \nthat offers a substantial life cover at affordable premiums, ensuring your family’s ﬁnancial \nstability when they need it the most. Unlike traditional life insurance, term plans focus solely on \nproviding life cove

In [11]:
print(documents[1].page_content)

Purchasing a term insurance plan is one of the most responsible ﬁnancial decisions you can 
make. In a world of uncertainties, this small investment can make a world of difference when 
it matters most. Here’s why a term plan life insurance is important: 
maintain their standard of living and meet their long-term goals in your absence. 
Whether it’s funding your child’s education, supporting a family member’s marriage, or 
Financial Security for Your Family: 
A term insurance plan acts as a ﬁnancial safety net, ensuring that your family can 
28/06/2026, 11:15 Term Insurance | Term Insurance Plan | ABSLI 
ts and Protection. Explore the plan under All Insurance > Endowment Plans. 
ABC Home ABC Solutions Connect 
Aditya Birla Sun Life Insurance Company Limited 
https://lifeinsurance.adityabirlacapital.com/term-insurance/ 2/27 
 
 
 
Term Insurance 
 
What Is Term Insurance? 
Term Insurance is a type of life insurance policy that provides ﬁnancial protection to your loved 
ones in case ter

### embedding model

In [4]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

# Initialize the model (downloads automatically on first run)
model_name = "BAAI/bge-m3"
model_kwargs = {"device": "cpu"}  # Change to "cuda" if you have a Nvidia GPU
encode_kwargs = {"normalize_embeddings": True}

embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)


### vector DB

In [12]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="C:\E_DRIVE\Langraph-Refresher\database\insurance\chroma_db"
)

### query to DB

In [13]:
query = "What is term insurance?"

results = vectorstore.similarity_search(
    query=query,
    k=3
)

for idx, doc in enumerate(results, start=1):
    print("\n" + "="*50)
    print(f"Result {idx}")
    print("="*50)
    print(doc.page_content[:500])


Result 1
Term Insurance 
 
What Is Term Insurance? 
Term Insurance is a type of life insurance policy that provides ﬁnancial protection to your loved 
ones in case term policy your untimely demise during the policy term. It is a pure protection plan 
that offers a substantial life cover at affordable premiums, ensuring your family’s ﬁnancial 
stability when they need it the most. Unlike traditional life insurance, term plans focus solely on

Result 2
28/06/2026, 11:15 Term Insurance | Term Insurance Plan | ABSLI 
https://lifeinsurance.adityabirlacapital.com/term-insurance/ 1/30 
 
 
 
 
 
 
BUY  ONLINE  
 
 
Term Insurance 
 
What Is Term Insurance? 
Term Insurance is a type of life insurance policy that provides ﬁnancial protection to your loved 
ones in case term policy your untimely demise during the policy term. It is a pure protection plan 
that offers a substantial life cover at affordable premiums, ensuring your family’s ﬁnancial

Result 3
28/06/2026, 11:15 Term Insurance | Ter

In [20]:
Q = [{"What are the four main exclusions listed in the term insurance policy?": "Suicide Clause, Death Due to Criminal Activities, Death Under the Influence, and Participation in Hazardous Activities."}, 
     {"What are the premium rates for a 30-year-old non-smoker for a ₹1 crore sum assured for 30 years?": "For a 30-year-old non-smoker, the premium for a ₹1 crore sum assured for 30 years is ₹10,000–₹12,000 annually."}]

Q

[{'What are the four main exclusions listed in the term insurance policy?': 'Suicide Clause, Death Due to Criminal Activities, Death Under the Influence, and Participation in Hazardous Activities.'},
 {'What are the premium rates for a 30-year-old non-smoker for a ₹1 crore sum assured for 30 years?': 'For a 30-year-old non-smoker, the premium for a ₹1 crore sum assured for 30 years is ₹10,000–₹12,000 annually.'}]

In [24]:
query =list( Q[0].keys())[0]
query

'What are the four main exclusions listed in the term insurance policy?'

In [ ]:
Q[0].values()

dict_values(['Suicide Clause, Death Due to Criminal Activities, Death Under the Influence, and Participation in Hazardous Activities.'])

In [25]:
query = list(Q[0].keys())[0]

results = vectorstore.similarity_search(
    query=query,
    k=3
)

for idx, doc in enumerate(results, start=1):
    print("\n" + "="*50)
    print(f"Result {idx}")
    print("="*50)
    print(doc.page_content[:500])


Result 1
assault, the insurer typically denies the claim. This exclusion ensures that the policy is used 
ethically and responsibly. 
 Death Under the Inﬂuence 
Deaths resulting from incidents where the policyholder was under the inﬂuence of alcohol, 
drugs, or other intoxicating substances are generally excluded. This clause is included to 
discourage reckless behaviour and ensure responsible conduct. 
 Participation in Hazardous Activities

Result 2
family. 
4. Buying a Home or Taking a Loan 
Major ﬁnancial commitments like home loans or business investments demand adequate 
ﬁnancial protection. 
 Debt Coverage: A term insurance plan ensures that your family isn’t left with the burden of 
repaying debts in your absence. It provides a ﬁnancial cushion during challenging times. 
 Sufficient Sum Assured: Ensure the sum assured matches or exceeds your total liabilities to

Result 3
while reducing your taxable income. 
Exclusions 
 Suicide Clause 
Most term insurance policies include a s

In [30]:
query =list( Q[1].keys())[0]
query

'What are the premium rates for a 30-year-old non-smoker for a ₹1 crore sum assured for 30 years?'

In [29]:
query = list(Q[1].keys())[0]

results = vectorstore.similarity_search(
    query=query,
    k=3
)

for idx, doc in enumerate(results, start=1):
    print("\n" + "="*50)
    print(f"Result {idx}")
    print("="*50)
    print(doc.page_content[:500])


Result 1
₹3 crore sum assured for 30 years: ₹22,000–₹25,000 annually. 
 ₹5 crore sum assured for 30 years: ₹30,000–₹40,000 annually. 
2. Smoker: 
 ₹1 crore sum assured for 30 years: ₹18,000–₹20,000 annually. 
 ₹3 crore sum assured for 30 years: ₹35,000–₹40,000 annually. 
 ₹5 crore sum assured for 30 years: ₹55,000–₹60,000 annually. 
These rates may vary based on the insurer, rider options, and policyholder’s proﬁle. Comparing 
plans can help you choose the best term insurance in India.

Result 2
adventure sports pay higher premiums due to increased risk. 
Gender 
 Women Beneﬁt: Women often enjoy lower premiums due to longer life expectancy and lower 
risk proﬁles. 
Occupation 
 High-Risk Professions: Jobs like mining, construction, or aviation attract higher premiums due 
to occupational hazards. 
Sample Premium Rates 
Here’s an example of premium rates for a 30-year-old policyholder: 
1. Non-Smoker: 
 ₹1 crore sum assured for 30 years: ₹10,000–₹12,000 annually.

Result 3
Aditya Birla

In [32]:
# Returns a list of (Document, np.ndarray) tuples
results_with_vectors = vectorstore.similarity_search_with_vectors(
    query=query,
    k=3
)

for doc, vector in results_with_vectors:
    print(f"Content: {doc.page_content}")
    print(f"Raw Embedding Vector: {vector[:5]}... (Length: {len(vector)})")

Content: ₹3 crore sum assured for 30 years: ₹22,000–₹25,000 annually. 
 ₹5 crore sum assured for 30 years: ₹30,000–₹40,000 annually. 
2. Smoker: 
 ₹1 crore sum assured for 30 years: ₹18,000–₹20,000 annually. 
 ₹3 crore sum assured for 30 years: ₹35,000–₹40,000 annually. 
 ₹5 crore sum assured for 30 years: ₹55,000–₹60,000 annually. 
These rates may vary based on the insurer, rider options, and policyholder’s proﬁle. Comparing 
plans can help you choose the best term insurance in India.
Raw Embedding Vector: [-0.01714182  0.00395696 -0.01483296 -0.03636777 -0.00199518]... (Length: 1024)
Content: adventure sports pay higher premiums due to increased risk. 
Gender 
 Women Beneﬁt: Women often enjoy lower premiums due to longer life expectancy and lower 
risk proﬁles. 
Occupation 
 High-Risk Professions: Jobs like mining, construction, or aviation attract higher premiums due 
to occupational hazards. 
Sample Premium Rates 
Here’s an example of premium rates for a 30-year-old policyholder: 


In [ ]:
# Returns a list of (Document, float) tuples
results_with_scores = vectorstore.similarity_search_with_score(
    query=query,
    k=3
)

for doc, score in results_with_scores:
    # Note: Chroma uses L2/Euclidean distance by default.
    # A LOWER score means the text is MORE similar to your query.
    print(f"Distance Score: {score}") 
    print(f"Content: {doc.page_content}\n")

In [33]:
# Fetch all items (or a limited number using the limit parameter)
db_data = vectorstore.get(limit=10)

# Extract the IDs list
all_ids = db_data["ids"]
print("First 5 IDs in the DB:", all_ids[:5])

First 5 IDs in the DB: ['bb316989-a47e-4151-b619-31ea6f969f9b', 'da4be5ac-6321-43d7-ad95-37933ac2cfc9', '07a12dab-becf-4088-8d45-1913d67b05b9', 'f40cb54a-65e9-4519-8859-15ac67af84a5', '30af975e-eb0f-4cb5-b081-96725f27a602']


In [ ]:
# Pass a list containing the ID(s) you want to fetch
target_ids = ['bb316989-a47e-4151-b619-31ea6f969f9b', 'da4be5ac-6321-43d7-ad95-37933ac2cfc9']

documents = vectorstore.get_by_ids(target_ids)

for doc in documents:
    print(doc)

In [ ]:
# Generate clean, predictable IDs (e.g., "insurance_policy_chunk_0")
custom_ids = [f"insurance_policy_chunk_{i}" for i in range(len(chunks))]

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=custom_ids, # <-- Pass them here!
    persist_directory="C:\E_DRIVE\Langraph-Refresher\database\insurance\chroma_db"
)

### CURD operations in DB

In [ ]:
import os
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings  # Replace with your actual embedding model

# 1. Setup Embeddings and Directory
embeddings = OpenAIEmbeddings()
persist_dir = r"C:\E_DRIVE\Langraph-Refresher\database\insurance\chroma_db"

# Sample initial data chunks
chunks = [
    Document(page_content="Car insurance covers accidental damages.", metadata={"category": "auto"}),
    Document(page_content="Health insurance covers hospital stays.", metadata={"category": "health"}),
]


# ==========================================
# C - CREATE (Initialize DB with Custom IDs)
# ==========================================
# Creating predictable keys based on chunk index
initial_ids = [f"doc_chunk_{i}" for i in range(len(chunks))]

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=initial_ids,
    persist_directory=persist_dir
)
print(f"--- CREATE: Added {len(initial_ids)} documents to Chroma. ---")


# ==========================================
# R - READ (Retrieve and Inspect IDs/Data)
# ==========================================
# Option A: Get all content and IDs present in the DB
db_contents = vectorstore.get()
print("\n--- READ (All IDs in DB) ---")
print("IDs currently in DB:", db_contents["ids"])

# Option B: Query via similarity search to inspect metadata tracking
query_results = vectorstore.similarity_search("Tell me about hospital coverage", k=1)
for doc in query_results:
    print(f"Search Result Text: '{doc.page_content}'")
    # LangChain injects Chroma's tracking ID inside doc.metadata["id"] implicitly during a search
    print(f"Search Result tracking ID: {doc.metadata.get('id')}")


# ==========================================
# U - UPDATE (Modify an existing document)
# ==========================================
# Let's update the text for the auto insurance chunk ('doc_chunk_0')
updated_document = Document(
    page_content="Car insurance covers accidental damages AND windshield replacements.",
    metadata={"category": "auto", "updated": "true"}
)

vectorstore.update_document(
    document_id="doc_chunk_0", 
    document=updated_document
)
print("\n--- UPDATE: Modified 'doc_chunk_0' successfully. ---")

# Verify the update happened
check_update = vectorstore.get(ids=["doc_chunk_0"])
print("Updated content looks like:", check_update["documents"])


# ==========================================
# D - DELETE (Remove specific documents)
# ==========================================
# Delete the health insurance chunk ('doc_chunk_1')
vectorstore.delete(ids=["doc_chunk_1"])
print("\n--- DELETE: Removed 'doc_chunk_1' from DB. ---")

# Verify current count remaining in DB
remaining_count = vectorstore._collection.count()
print(f"Total documents left in collection: {remaining_count}")

In [ ]:
# HR Collection
hr_store = Chroma(
    collection_name="hr_documents",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db"
)

# Finance Collection
finance_store = Chroma(
    collection_name="finance_records",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db"
)

# OpenAI Collection (e.g., 1536 dimensions)
openai_store = Chroma(collection_name="openai_data", embedding_function=openai_embeds, ...)

# HuggingFace Collection (e.g., 384 dimensions)
hf_store = Chroma(collection_name="huggingface_data", embedding_function=hf_embeds, ...)

In [37]:
documents[1].metadata

{'producer': 'Microsoft® Word 2021',
 'creator': 'Microsoft® Word 2021',
 'creationdate': '2026-06-28T11:56:25+05:30',
 'title': 'Term Insurance | Term Insurance Plan | ABSLI',
 'author': 'smrit',
 'moddate': '2026-06-28T11:56:25+05:30',
 'source': 'resources/Term Insurance _clean.pdf',
 'total_pages': 27,
 'page': 1,
 'page_label': '2'}

In [ ]:
# ====================================================================
# USE CASE B: Filtered Search (Only HR Policies)
# ====================================================================
print("\n--- Filtered Search Results (HR Only) ---")
filtered_results = vector_store.similarity_search(
    query_text, 
    k=2, 
    filter={"source": "hr_policy"} # <--- Chroma ignores the tweets entirely!
)

### Evaluation Prompts

In [ ]:
assesment_list1 = {
    "What is term insurance?": "Term insurance is a type of life insurance policy that provides financial protection to your loved ones in case of your untimely demise during the policy term. It offers a substantial life cover at affordable premiums and focuses solely on providing life coverage without investment or savings components.",
    "Who should buy term insurance?": "Term insurance is suitable for anyone who wants to secure their family’s financial future, including sole breadwinners, young professionals, parents, homemakers, and business owners.",
    "What are the main benefits of term insurance?": "Key benefits include affordable premiums, debt protection, critical illness coverage, tax benefits, high sum assured at low cost, flexible policy terms, and customizable add-ons (riders).",
    "How does a term insurance plan work?": "You choose a plan and coverage amount, pay regular premiums, and if you pass away during the policy term, your nominee receives the sum assured as a death benefit. There is no maturity benefit unless you opt for a Return of Premium (ROP) feature.",
    "What payout options are available for the death benefit?": "You can choose from a lump sum payout, monthly income, or a combination of both, depending on your family’s needs.",
    "What factors affect term insurance premiums?": "Premiums are influenced by age, health and medical history, lifestyle habits (like smoking), gender, occupation, and the sum assured.",
    "What are common inclusions and exclusions in term insurance?": "Inclusions: death benefit, terminal illness benefit, and optional riders (like accidental death benefit). Exclusions: suicide within the first year, death due to criminal activities, death under the influence of intoxicants, and participation in hazardous activities unless disclosed and covered.",
    "What documents and eligibility are required to buy term insurance?": "Eligibility: Minimum age 18, maximum up to 65 (varies by plan), policy term 5–40 years or up to a specific age. Documents: identity proof, address proof, age proof, income proof, photographs, and medical records if required.",
    "How can you make term insurance more affordable?": "Start early to lock in lower premiums, choose online plans for reduced costs, and compare different plans to find the best value for your needs.",
    "How does term insurance differ from whole life insurance?": "Term insurance provides coverage for a specific period with no cash value or investment component, while whole life insurance covers you for life, accumulates cash value, and may allow policy loans."
}



In [ ]:
assesment_list2 = {
    "What is the minimum and maximum entry age for ABSLI term insurance plans?": "The minimum entry age is 18 years, and the maximum is generally up to 65 years (varies by plan and insurer).",
    "What are the premium rates for a 30-year-old non-smoker for a ₹1 crore sum assured for 30 years?": "For a 30-year-old non-smoker, the premium for a ₹1 crore sum assured for 30 years is ₹10,000–₹12,000 annually.",
    "What payout options does ABSLI offer for the death benefit?": "ABSLI offers lump sum, regular income, or a combination of both as payout options for the death benefit.",
    "What is the Return of Premium (ROP) feature in term insurance?": "The ROP feature refunds all premiums paid if the policyholder survives the policy term, but it comes at a higher premium cost.",
    "What are the main exclusions in ABSLI term insurance policies?": "Exclusions include suicide within the first year, death due to criminal activities, death under the influence of intoxicants, and participation in hazardous activities unless disclosed and covered.",
    "What documents are required to apply for a term insurance plan?": "Required documents include identity proof (Aadhaar, PAN, Passport, Voter ID), address proof, age proof (birth certificate, passport, school leaving certificate), income proof (salary slips, ITR, bank statements), recent photographs, and medical records if applicable.",
    "What is the sum assured to premium ratio recommended in the document?": "The document recommends a sum assured that is 10–20 times your annual income for adequate family protection.",
    "What is the critical illness rider in ABSLI term insurance?": "The critical illness rider provides a lump-sum payout upon diagnosis of specified illnesses like cancer, heart attack, or kidney failure, helping cover treatment and related expenses.",
    "How does ABSLI support digital ease for customers?": "ABSLI allows customers to compare plans, apply, pay premiums, and file claims entirely online, making the process seamless and convenient.",
    "What is the Enhanced Life Stage Protection feature?": "Enhanced Life Stage Protection allows you to increase your life cover at significant milestones such as marriage or the birth of a child, ensuring your policy adapts to your changing responsibilities."
}



In [ ]:
assesment_factual = {
    "What is term insurance?": "Term insurance is a type of life insurance policy that provides financial protection to your loved ones in case of your untimely demise during the policy term, offering substantial life cover at affordable premiums without investment or savings components.",
    "Who should consider buying a term insurance plan?": "Sole breadwinners, young professionals, parents, homemakers, and business owners should consider buying a term insurance plan to secure their family’s financial future.",
    "What is the minimum entry age for ABSLI term insurance?": "The minimum entry age is 18 years at the time of policy purchase.",
    "What is the maximum entry age for ABSLI term insurance?": "The maximum entry age is generally up to 65 years, depending on the plan.",
    "What is the typical policy term range for ABSLI term insurance?": "The policy term typically ranges from 5 to 40 years or up to a specific age, such as 75-100 years.",
    "What payout options are available for the death benefit?": "Payout options include lump sum, monthly income, or a combination of both, as per the policyholder’s selection.",
    "Does term insurance offer a maturity benefit if the policyholder survives the term?": "No, term insurance does not offer a maturity benefit unless the Return of Premium (ROP) feature is selected, which refunds all premiums paid if no claim is made.",
    "What is the critical illness rider in ABSLI term insurance?": "The critical illness rider provides a lump-sum payout upon diagnosis of specified illnesses like cancer, heart attack, or kidney failure.",
    "What are the main exclusions in ABSLI term insurance policies?": "Exclusions include suicide within the first year, death due to criminal activities, death under the influence of intoxicants, and participation in hazardous activities unless disclosed and covered.",
    "What documents are required to apply for a term insurance plan?": "Required documents include identity proof (Aadhaar, PAN, Passport, Voter ID), address proof, age proof (birth certificate, passport, school leaving certificate), income proof (salary slips, ITR, bank statements), recent photographs, and medical records if applicable.",
    "What is the recommended sum assured to premium ratio?": "The document recommends a sum assured that is 10–20 times your annual income for adequate family protection.",
    "What are the premium rates for a 30-year-old non-smoker for a ₹1 crore sum assured for 30 years?": "For a 30-year-old non-smoker, the premium for a ₹1 crore sum assured for 30 years is ₹10,000–₹12,000 annually.",
    "How does ABSLI support digital ease for customers?": "ABSLI allows customers to compare plans, apply, pay premiums, and file claims entirely online.",
    "What is the Enhanced Life Stage Protection feature?": "Enhanced Life Stage Protection allows you to increase your life cover at significant milestones such as marriage or the birth of a child.",
    "What is the difference between term insurance and whole life insurance regarding cash value?": "Term insurance has no cash value accumulation, while whole life insurance accumulates cash value over time.",
    "What is the suicide clause in ABSLI term insurance?": "If the policyholder commits suicide within the first year of policy commencement, the insurer may not pay the full death benefit; only premiums paid may be refunded, depending on policy terms.",
    "What is the waiver of premium rider?": "The waiver of premium rider continues your policy coverage without requiring premium payments in case of disability or critical illness.",
    "What are the tax benefits of term insurance?": "Premiums paid are eligible for tax deductions under Section 80C of the Income Tax Act, 1961, up to ₹1.5 lakh annually, and the death benefit is generally tax-exempt under Section 10(10D).",
    "What is the joint life protection feature?": "Certain ABSLI term plans offer joint life coverage, allowing both spouses to be insured under a single policy.",
    "What is the Return of Premium (ROP) option?": "The ROP option refunds all premiums paid if the policyholder survives the policy term, providing both protection and savings."
}



In [14]:
claud_ass = {
    "What is term insurance?": "Term Insurance is a type of life insurance policy that provides financial protection to your loved ones in case of your untimely demise during the policy term. It is a pure protection plan that offers a substantial life cover at affordable premiums, ensuring your family's financial stability when they need it the most.",
    "Under which section of the Income Tax Act are term insurance premiums eligible for tax deductions?": "Section 80C of the Income Tax Act, 1961, up to the maximum limit (up to ₹1.5 lakh annually).",
    "Under which section is the death benefit received by the nominee generally exempt from taxes?": "Section 10(10D) of the Income Tax Act, 1961.",
    "What are the three payout options available for the death benefit in a term insurance plan?": "Lump Sum (entire sum assured paid as a one-time amount), Monthly Income (payout distributed as regular monthly income), and Combination (part paid as lump sum and rest distributed as monthly income).",
    "What is the Return of Premium (ROP) feature in term insurance?": "An optional feature where all the premiums you paid are refunded at the end of the policy term if you survive the policy term.",
    "What sum assured can be secured with a nominal premium according to the document?": "A sum assured that is 10-20 times your annual income.",
    "What is the minimum age eligibility for a term insurance plan according to the document?": "18 years at the time of policy purchase.",
    "What is the maximum age eligibility for a term insurance plan according to the document?": "Generally up to 65 years (varies by insurer and plan).",
    "What are the sample annual premium rates for a non-smoker 30-year-old with a ₹1 crore sum assured for 30 years?": "₹10,000–₹12,000 annually.",
    "What are the sample annual premium rates for a smoker 30-year-old with a ₹5 crore sum assured for 30 years?": "₹55,000–₹60,000 annually.",
    "What is the suicide clause in term insurance?": "If the policyholder commits suicide within the first year of the policy commencement, the insurer may not pay the full death benefit. In such cases, only the premiums paid may be refunded, depending on the policy terms.",
    "What does the Waiver of Premium Rider do?": "It waives off future premiums if the policyholder becomes disabled or critically ill and cannot pay, ensuring the policy remains active.",
    "What is the ABSLI DigiShield Plan's maximum coverage age?": "Life cover extending up to 100 years of age.",
    "How many distinct plan options does the ABSLI DigiShield Plan offer?": "10 distinct plan options, including Level Cover, Increasing Cover, and Whole Life Cover.",
    "What general rule is recommended for calculating the appropriate sum assured?": "Aim for coverage that is at least 10-15 times your annual income.",
    "What is the typical policy term range for term insurance plans?": "Typically ranges from 5 to 40 years or up to a specific age (e.g., 75-100 years).",
    "What does the Critical Illness Rider provide?": "A lump-sum payout upon diagnosis of specified life-threatening illnesses like cancer, heart attack, or kidney failure.",
    "What is the key difference between term insurance and whole life insurance regarding cash value?": "Term insurance has no cash value accumulation, while whole life insurance accumulates cash value over time.",
    "What documents are required as income proof for a term insurance application?": "Salary Slips (Last 3 Months), Income Tax Returns, and Bank Statements.",
    "What does the Enhanced Life Stage Protection feature in ABSLI term plans allow?": "It allows you to increase your life cover at significant milestones like marriage or the birth of a child.",
    "What is the Accidental Death Benefit Rider?": "It provides an additional payout if the policyholder dies due to an accident.",
    "What are the flexible policy term options available in term insurance?": "Coverage ranging from 10 to 40 years or a policy that extends until a specific age, such as 60, 70, or even 80 years.",
    "What is the Terminal Illness Benefit in term insurance?": "A benefit where the policyholder receives an early payout if diagnosed with a terminal condition, helping cover medical treatment expenses.",
    "What does joint life protection in ABSLI term plans offer?": "The option to insure both yourself and your spouse under a single policy, ensuring comprehensive family protection and financial security to both partners.",
    "What strategy does the document recommend to make term insurance more affordable?": "Start early by purchasing a policy in your 20s or 30s to lock in lower premiums and maximise coverage, and choose online plans as digital plans often come with reduced operational costs translating to lower premiums."
}

In [ ]:
claud_fact = {
    "What are the four main exclusions listed in the term insurance policy?": "Suicide Clause, Death Due to Criminal Activities, Death Under the Influence, and Participation in Hazardous Activities.",
    "What happens if a policyholder dies while engaging in illegal or criminal activities?": "The insurer typically denies the claim. This exclusion ensures that the policy is used ethically and responsibly.",
    "What high-risk activities are mentioned as examples under the hazardous activities exclusion?": "Skydiving, bungee jumping, motor racing, or other adventurous pursuits.",
    "What is the claim settlement ratio (CSR) and why is it important?": "The claim settlement ratio is a critical indicator of an insurer's reliability. A high CSR means the insurer processes and settles a majority of claims, ensuring your family receives the promised benefit without delays or complications.",
    "What are the five important life stages mentioned in the document to buy term life insurance?": "Starting Your Career, Getting Married, Becoming a Parent, Buying a Home or Taking a Loan, and Planning for Retirement.",
    "What is the recommended minimum policy term if you have a home loan of 20 years?": "Choose a term of at least 20 years to cover the repayment period.",
    "Until what age should a term policy ideally last to cover working years?": "Until your planned retirement age, typically 60–65 years.",
    "What are the two ABSLI term insurance plans specifically mentioned as best plans in India 2025?": "ABSLI DigiShield Plan and ABSLI Salaried Term Plan.",
    "What multiple plan options does the ABSLI Salaried Term Plan include?": "Life Cover, Life Cover with Return of Premium, Fixed Income Cover, and Increasing Income Cover.",
    "What additional riders does the ABSLI Salaried Term Plan offer?": "Accidental Death and Disability Rider and Waiver of Premium Rider.",
    "What factors determine the premium amount in a term insurance plan?": "Age, health, sum assured, policy term, and any additional riders selected.",
    "Why do women often enjoy lower premiums in term insurance?": "Due to longer life expectancy and lower risk profiles.",
    "What occupations are mentioned as attracting higher premiums due to occupational hazards?": "Jobs like mining, construction, or aviation.",
    "What is the Accidental Disability Rider?": "It provides a payout if the policyholder becomes permanently disabled due to an accident.",
    "What are the Surgical Care and Hospital Care Riders?": "Riders that cover hospitalisation and surgical expenses, reducing the financial burden during medical emergencies.",
    "What premium payment frequency options are available under ABSLI term insurance plans?": "Monthly, quarterly, half-yearly, yearly, or even a one-time single premium payment.",
    "What is the staggered payment with increasing annual income payout option?": "It offers a growing income stream to keep pace with inflation and rising expenses.",
    "What documents are accepted as identity proof for a term insurance application?": "Aadhaar Card, PAN Card, Passport, and Voter ID.",
    "What documents are accepted as age proof for a term insurance application?": "Birth Certificate, Passport, and School Leaving Certificate.",
    "What does the document state about deaths resulting from being under the influence of alcohol or drugs?": "Deaths resulting from incidents where the policyholder was under the influence of alcohol, drugs, or other intoxicating substances are generally excluded, as this clause is included to discourage reckless behaviour and ensure responsible conduct."
}